# Notebook 13: Polymorphism

**Polymorphism** (Greek: 'many forms') lets you write code that works on a base class and automatically gets the right behaviour for each derived class — determined at **runtime**.

Instead of asking "what type is this object?", you just call the method and the object figures out what to do. This is the core of object-oriented design.

## The Problem Without `virtual`

Recall from the inheritance notebook: without `virtual`, the method called depends on the **pointer type**, not the actual object.

```cpp
Shape *s = new Rectangle(4, 3);
s->area();  // Calls Shape::area() — returns 0. Wrong!
```

The compiler binds `s->area()` to `Shape::area` at compile time because `s` is a `Shape *`. It does not matter that the object is actually a `Rectangle`.

This is called **static dispatch** (or early binding). We need **dynamic dispatch** (late binding) — decide which method to call at runtime based on the actual object type. That is what `virtual` provides.

## The `virtual` Keyword

Add `virtual` to the base class method declaration. The derived class provides an overriding implementation with the **same signature**.

Now calling the method through a base pointer uses the actual object's type to find the right function.

In [ ]:
#include <iostream>
#include <string>

class Shape {
public:
    std::string name;

    Shape(const std::string &n) : name(n) {}

    virtual double area() const {
        return 0.0;
    }

    virtual ~Shape() {}  // Always make the destructor virtual in a base class!
};

In [ ]:
#include <iostream>

class Rectangle : public Shape {
public:
    double width;
    double height;

    Rectangle(double w, double h) : Shape("Rectangle"), width(w), height(h) {}

    double area() const {
        return width * height;
    }
};

class Circle : public Shape {
public:
    double radius;

    Circle(double r) : Shape("Circle"), radius(r) {}

    double area() const {
        return 3.14159 * radius * radius;
    }
};

// Now the dispatch is correct:
Shape *s1 = new Rectangle(4.0, 3.0);
Shape *s2 = new Circle(5.0);

std::cout << s1->name << " area: " << s1->area() << std::endl;  // 12
std::cout << s2->name << " area: " << s2->area() << std::endl;  // ~78.5

delete s1;
delete s2;

**Exercise 1:** Using the `Shape`, `Rectangle`, and `Circle` classes defined above, create an array of three `Shape` pointers — mix of `Rectangle` and `Circle` objects. Loop through the array, print each shape's name and area, then delete each pointer.

In [ ]:
// Your code here

## How It Works: The vtable

When a class has `virtual` methods, the compiler creates a **vtable** (virtual function table) — an array of function pointers, one per virtual method.

```
Shape vtable:      [0] → Shape::area
Rectangle vtable:  [0] → Rectangle::area
Circle vtable:     [0] → Circle::area

Shape object:     [ vptr | name ]      vptr → Shape vtable
Rectangle object: [ vptr | name | w | h ]  vptr → Rectangle vtable
```

Every object of a class with virtual methods contains a hidden **vptr** pointing to its class's vtable.

When you call `s->area()`, the CPU:
1. Follows `s` to the object
2. Reads the `vptr` from the object
3. Looks up `area` in the vtable
4. Calls the function pointer found there

**Cost:** one extra pointer per object, one pointer indirection per virtual call. Usually negligible — only relevant in extremely tight inner loops.

## Pure Virtual Functions

A **pure virtual function** is declared with `= 0`:

```cpp
virtual double area() const = 0;
```

This means:
- The base class provides **no implementation** (or an optional one that must be called explicitly)
- Every concrete derived class **must** override it
- The base class becomes **abstract** — you cannot create instances of it directly

Pure virtual functions define the *contract* that derived classes must fulfil.

In [ ]:
#include <iostream>
#include <string>

class AbstractShape {
public:
    std::string name;

    AbstractShape(const std::string &n) : name(n) {}

    virtual double area() const = 0;       // pure virtual
    virtual double perimeter() const = 0;  // pure virtual

    virtual ~AbstractShape() {}
};

// AbstractShape s("test");  // COMPILE ERROR: cannot instantiate abstract class

class Rect2 : public AbstractShape {
public:
    double w, h;
    Rect2(double w, double h) : AbstractShape("Rectangle"), w(w), h(h) {}

    double area() const      { return w * h; }
    double perimeter() const { return 2 * (w + h); }
};

Rect2 r(5.0, 3.0);
std::cout << "Area: " << r.area() << ", Perimeter: " << r.perimeter() << std::endl;

## Abstract Classes

A class with at least one pure virtual function is **abstract**:
- You **cannot** create objects of it: `AbstractShape s;` → compile error
- You **can** have pointers and references to it: `AbstractShape *s = new Rect2(...);
- It serves as a contract — all concrete subclasses must implement the pure virtuals

Abstract classes are perfect for defining interfaces and polymorphic APIs.

**Exercise 2:** Using `AbstractShape` (defined above), create a `Circle2` class that implements both `area()` and `perimeter()` (use π ≈ 3.14159). Then create an array of `AbstractShape` pointers containing a mix of `Rect2` and `Circle2` objects. Loop through, print each shape's name, area, and perimeter. Clean up properly.

In [ ]:
// Your code here

## The Virtual Destructor

This is **critical** and a very common bug source.

If you delete a derived object through a base pointer and the base destructor is **not virtual**, the derived destructor is **not called** — leading to resource leaks or worse.

**Rule:** Any class intended to be a base class (i.e. you might delete via a base pointer) **must** have a `virtual` destructor.

In [ ]:
#include <iostream>

class BaseNonVirtual {
public:
    ~BaseNonVirtual() { std::cout << "BaseNonVirtual destructor" << std::endl; }
};

class DerivedLeak : public BaseNonVirtual {
public:
    int *data;
    DerivedLeak() : data(new int[100]) {}
    ~DerivedLeak() {
        std::cout << "DerivedLeak destructor (frees data)" << std::endl;
        delete[] data;  // This is NEVER called when deleting via BaseNonVirtual*!
    }
};

std::cout << "--- Non-virtual destructor (BAD): ---" << std::endl;
BaseNonVirtual *bad = new DerivedLeak();
delete bad;  // Only calls ~BaseNonVirtual(). Memory leak!

// FIX: make the base destructor virtual
std::cout << std::endl << "(In well-designed code, Shape/AbstractShape already has virtual ~)" << std::endl;

## Polymorphism in Practice

The real power of polymorphism: write a function once that works with any derived type, including types that don't exist yet when you write the function.

In [ ]:
#include <iostream>

// This function works for ANY class derived from AbstractShape
void processShape(const AbstractShape &s) {
    std::cout << s.name
              << " | area: " << s.area()
              << " | perimeter: " << s.perimeter()
              << std::endl;
}

Rect2 myRect(6.0, 4.0);
// Assuming Circle2 was defined in Exercise 2 — if not, use another Rect2
Rect2 myRect2(3.0, 3.0);

processShape(myRect);
processShape(myRect2);
// processShape(myCircle);  // Works too, once Circle2 is defined

## Interface Pattern

A class with **only** pure virtual functions (no data, no implementation) is called an **interface**. It defines a contract that implementing classes must fulfil.

C++ does not have a built-in `interface` keyword (unlike Java/C#), but the pattern is widely used.

A class can inherit from multiple interfaces — this is a common and safe use of multiple inheritance.

In [ ]:
#include <iostream>
#include <string>

class Printable {
public:
    virtual void print() const = 0;
    virtual ~Printable() {}
};

class Resizable {
public:
    virtual void resize(double factor) = 0;
    virtual ~Resizable() {}
};

// Implements both interfaces
class Box : public Printable, public Resizable {
public:
    double side;
    Box(double s) : side(s) {}

    void print() const {
        std::cout << "Box with side " << side << std::endl;
    }

    void resize(double factor) {
        side *= factor;
    }
};

Box b(5.0);
b.print();
b.resize(2.0);
b.print();

// Can also use via interface pointer
Printable *p = &b;
p->print();

**Exercise 3:** Create an interface class `Serializable` with a pure virtual method `std::string serialize() const`. Implement it in a `Point` class (with `x` and `y` double members — serialize to `"Point(x,y)"`) and in `Rect2` by adding the interface to its inheritance list. Write a free function `void printSerialized(const Serializable *obj)` that calls `serialize()` and prints the result.

In [ ]:
// Your code here

## Final Exercise

Create a mini drawing system:

1. Abstract base class `Drawable` with pure virtual `void draw() const` and pure virtual `double area() const`. Virtual destructor.
2. `Square` — side length. `draw()` prints `"Drawing a square with side [s]"`. `area()` = s².
3. `Triangle` — base and height. `draw()` prints `"Drawing a triangle with base [b] height [h]"`. `area()` = 0.5 * base * height.
4. `Circle3` — radius. `draw()` prints `"Drawing a circle with radius [r]"`. `area()` = π * r².
5. Create an array of 5 `Drawable` pointers with a mix of the three types.
6. Loop through and call `draw()` and `area()` on each.
7. Delete all pointers properly at the end.

In [ ]:
// Your code here

## Modern C++ (C++11 and Beyond)

### `override`
Place `override` after the method signature in the derived class. The compiler verifies you are actually overriding a virtual method — catches typos and signature mismatches that would silently create a new method instead of overriding.

### `final`
- On a class: `class Concrete final` — no further inheritance allowed
- On a method: `void draw() const override final` — no further override allowed

### `nullptr`
In C++11, use `nullptr` instead of `NULL` or `0` for null pointers. It is type-safe and unambiguous.

In [ ]:
#include <iostream>
#include <string>

class DrawableModern {
public:
    virtual void draw() const = 0;
    virtual ~DrawableModern() {}
};

class SquareModern : public DrawableModern {
public:
    double side;
    SquareModern(double s) : side(s) {}

    void draw() const override {  // 'override' ensures this truly overrides a virtual
        std::cout << "Square side=" << side << std::endl;
    }
    // void drow() const override {}  // Would be a COMPILE ERROR — no 'drow' in base
};

class SealedSquare final : public SquareModern {
public:
    SealedSquare(double s) : SquareModern(s) {}
    void draw() const override final {
        std::cout << "[sealed] Square side=" << side << std::endl;
    }
};

// nullptr instead of NULL or 0
DrawableModern *ptr = nullptr;
ptr = new SquareModern(3.0);
ptr->draw();
delete ptr;
ptr = nullptr;